In [1]:
import json
import random
import statistics

RESULTS_PATH = "/kaggle/input/datasets/teslaincarnate/tester/eval_results.json"

try:
    with open(RESULTS_PATH, "r") as f:
        results = json.load(f)
except FileNotFoundError:
    print(f"❌ Could not find {RESULTS_PATH}")
    exit()

perfect_scores = []
failed_scores = []

for r in results:
    if r.get("score") is not None:
        if r["score"] == r["max_score"]:
            perfect_scores.append(r)
        elif r["score"] < r["max_score"]:
            failed_scores.append(r)

perfect_lengths = [len(str(r.get("model_response", ""))) for r in perfect_scores]
failed_lengths = [len(str(r.get("model_response", ""))) for r in failed_scores]

avg_perfect_len = statistics.mean(perfect_lengths) if perfect_lengths else 0
avg_failed_len = statistics.mean(failed_lengths) if failed_lengths else 0

print("\n" + "="*60)
print("🔍 VERBOSITY BIAS ANALYSIS")
print("="*60)
print(f"Average length of PERFECT (100%) responses: {avg_perfect_len:.1f} characters")
print(f"Average length of FAILED (deducted) responses:  {avg_failed_len:.1f} characters")

if avg_perfect_len > avg_failed_len:
    print("\n🚨 CONCLUSION: The judge exhibits Verbosity Bias.")
    print("It systematically awards higher scores to longer answers, penalizing conciseness.")
else:
    print("\n✅ CONCLUSION: No obvious Verbosity Bias detected in character length.")

failed_mcqs = [r for r in failed_scores if r.get("category") == "multiple_choice"]
failed_clinical = [r for r in failed_scores if r.get("category") != "multiple_choice"]

def print_audit(sample_list, sample_size=3, title=""):
    print(f"\n{'='*60}\n{title}\n{'='*60}")
    if not sample_list:
        print("No failed examples found in this category.")
        return
        
    samples = random.sample(sample_list, min(sample_size, len(sample_list)))
    
    for i, r in enumerate(samples):
        print(f"\n🚨 FAILURE EXAMPLE {i+1} (Layer {r['layer']})")
        print(f"QUESTION: {r['question']}")
        print(f"--------------------------------------------------")
        print(f"GOLD ANSWER: {r['gold_answer']}")
        print(f"MODEL RESPONSE: {r['model_response']}")
        print(f"--------------------------------------------------")
        print(f"JUDGE GAVE: {r['score']}/{r['max_score']}")
        print(f"JUDGE REASONING: {r['judge_reasoning']}")

print_audit(failed_mcqs, sample_size=2, title="AUDITING MULTIPLE CHOICE FAILURES")
print_audit(failed_clinical, sample_size=2, title="AUDITING CLINICAL REASONING FAILURES")


🔍 VERBOSITY BIAS ANALYSIS
Average length of PERFECT (100%) responses: 1544.1 characters
Average length of FAILED (deducted) responses:  1556.3 characters

✅ CONCLUSION: No obvious Verbosity Bias detected in character length.

AUDITING MULTIPLE CHOICE FAILURES

🚨 FAILURE EXAMPLE 1 (Layer 2)
QUESTION: A boy presented with multiple non suppurative osteomyelitis with sickle cell anaemia. What will be the causative organism?
--------------------------------------------------
GOLD ANSWER: Salmonella
MODEL RESPONSE: In a patient with sickle cell anemia (SCA) presenting with multiple non-suppurative osteomyelitis, the causative organism is most likely Staphylococcus aureus. This is due to the unique pathophysiology of SCA, which predisposes individuals to recurrent bacterial infections, particularly those caused by encapsulated organisms.

Sickle cell anemia is a genetic disorder characterized by the production of abnormal hemoglobin (HbS), which polymerizes under deoxygenated conditions, lea